In [1]:
import torch
from imutils import paths
import numpy as np

In [2]:
# Remove the sample_data folder from google colab
!rm -rf /content/*
!git clone https://github.com/GiaHuyPham/DeepLearning_ComputerVision.git

Cloning into 'DeepLearning_ComputerVision'...
remote: Enumerating objects: 3034, done.
remote: Counting objects: 100% (779/779), done.
remote: Compressing objects: 100% (765/765), done.
remote: Total 3034 (delta 46), reused 13 (delta 13), pack-reused 2255 (from 3)
Receiving objects: 100% (3034/3034), 133.03 MiB | 48.14 MiB/s, done.
Resolving deltas: 100% (172/172), done.


# Restructure the File tree

In [3]:
import shutil
import os

""" Clean up the file tree"""

# Create new folder for the train data
os.makedirs("/content/data",exist_ok=True)
os.makedirs("/content/unseen_data",exist_ok=True)
# Create saving path
os.makedirs("/content/output",exist_ok=True)
os.makedirs("/content/output/model",exist_ok=True)
os.makedirs("/content/output/plots",exist_ok=True)
os.makedirs("/content/output/pred_masks",exist_ok=True)

# Move the image folder and mask folder to new folder
shutil.move("/content/DeepLearning_ComputerVision/images","/content/data/images")
shutil.move("/content/DeepLearning_ComputerVision/images_unseen","/content/unseen_data/images_unseen")
shutil.move("/content/DeepLearning_ComputerVision/masks","/content/data/masks")
shutil.move("/content/DeepLearning_ComputerVision/masks_unseen","/content/unseen_data/masks_unseen")
shutil.move("/content/DeepLearning_ComputerVision/unet_model.pth","/content/output/model")

shutil.move("/content/DeepLearning_ComputerVision/pyunet","/content/pyunet")

# Delete unnessary files and folders
shutil.rmtree("/content/DeepLearning_ComputerVision")

# Check if folders are there?
print(os.listdir('/content/data'))

['masks', 'images']


# Device Agnostic

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load and Create an Instance for the UNET Model

In [6]:
MODEL_PATH = "/content/output/model"
unet_model = torch.load(os.path.join(MODEL_PATH,"unet_model.pth"),weights_only=False)
unet_model.to(device)


Unet(
  (encoder): Encoder(
    (encBlocks): ModuleList(
      (0): Block(
        (block): Sequential(
          (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1))
          (1): ReLU()
          (2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1))
        )
      )
      (1): Block(
        (block): Sequential(
          (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
          (1): ReLU()
          (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1))
        )
      )
      (2): Block(
        (block): Sequential(
          (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
          (1): ReLU()
          (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
        )
      )
    )
    (maxPool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (decoder): Decoder(
    (upconv): ModuleList(
      (0): ConvTranspose2d(64, 32, kernel_size=(2, 2), stride=(2, 2))
      (1): ConvTranspose2d(32, 16, kernel_size=(2, 2), stride=(2, 2))
    )

# Implement Model on Unseen Data

In [10]:
# Predict Masks
from PIL import Image
import cv2
from torchvision import transforms
from torchvision.transforms import InterpolationMode


# Define image input shape
INPUT_IMAGE_HEIGHT = 128
INPUT_IMAGE_WIDTH = 128

# Write a transform for image
image_transform = transforms.Compose([
    # Transfrom to PILimage
    transforms.ToPILImage(),
    # Resize our image
    transforms.Resize(size=(INPUT_IMAGE_HEIGHT,INPUT_IMAGE_WIDTH)),
    # Turn image into tensor
    transforms.ToTensor()
])

mask_transform = transforms.Compose([
    # Transfrom to PILimage
    transforms.ToPILImage(),
    # Resize our image
    transforms.Resize(size=(INPUT_IMAGE_HEIGHT,INPUT_IMAGE_WIDTH),
                      interpolation=InterpolationMode.NEAREST),
    # Turn image into tensor
    transforms.ToTensor()
])

IMAGE_UNSEEN_DATASET = os.path.join("/content/unseen_data","images_unseen")
imagePath_unseen = sorted(list(paths.list_images(IMAGE_UNSEEN_DATASET)))
pred_masks_path = "/content/output/pred_masks"

for imgPath in imagePath_unseen:
  image_name = os.path.basename(imgPath)

  img = cv2.imread(imgPath,0)
  img = image_transform(img)
  img = img.unsqueeze(dim=1)
  img = img.to(device)

  unet_model.eval()
  with torch.inference_mode():
    mask_logits = unet_model(img).squeeze()
    pred_mask = torch.sigmoid(mask_logits)
    pred_mask = pred_mask.cpu().numpy()
    pred_mask = (pred_mask>0.5)*255
    pred_mask = pred_mask.astype(np.uint8)

  save_name = os.path.splitext(image_name)[0].replace('img_unseen', 'mask_unseen') + '.png'
  Image.fromarray(pred_mask).save(os.path.join(pred_masks_path, save_name))

  # print(f"Saved: {save_name}")

print("Done! All predicted masks saved.")

Done! All predicted masks saved.


## Plot

In [11]:
from pyunet.helperfunctions import make_frame_viewer
predMask_path = sorted(list(paths.list_images(pred_masks_path)))
make_frame_viewer(imagePath_unseen,predMask_path,image_transform,mask_transform)

interactive(children=(IntSlider(value=0, description='Frame:', max=379), Output()), _dom_classes=('widget-inte…

<function pyunet.helperfunctions.make_frame_viewer.<locals>.show_frame(iiframe)>